# TechMind - Treinamento do Modelo de ML

Treina um classificador **Logistic Regression + TF-IDF** para categorizar conteúdos técnicos em 8 categorias.

**Dataset:** `ml-service/data/train.csv` — 80 exemplos sintéticos (10 por categoria) em português.

**Saída:** `ml-service/model.joblib` (modelo + vectorizer empacotados)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
import nltk
import re
import joblib
import os

nltk.download('stopwords', quiet=True)
stopwords_pt = nltk.corpus.stopwords.words('portuguese')
print(f'{len(stopwords_pt)} stopwords carregadas')

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'train.csv')
MODEL_PATH = os.path.join('..', 'model.joblib')

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {len(df)} exemplos')
print(f'Categorias: {df["categoria"].value_counts().to_dict()}')

In [ ]:
def preprocess(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = re.sub(r'\d+', '', texto)
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_pt and len(t) > 2]
    return ' '.join(tokens)

df['texto_limpo'] = df['texto'].apply(preprocess)
print('Pré-processamento concluído')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['texto_limpo'], df['categoria'],
    test_size=0.2, random_state=42, stratify=df['categoria']
)
print(f'Treino: {len(X_train)} | Teste: {len(X_test)}')

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, multi_class='multinomial'))
])

pipeline.fit(X_train, y_train)
print('Modelo treinado')

In [ ]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Acurácia no holdout: {acc:.4f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
scores = cross_val_score(pipeline, df['texto_limpo'], df['categoria'], cv=5)
print(f'Cross-validation (5-folds): {scores}')
print(f'Média: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
def predict_with_threshold(texto, threshold=0.5):
    probs = pipeline.predict_proba([texto])[0]
    max_prob = probs.max()
    if max_prob < threshold:
        return 'Desconhecida', float(max_prob), []
    classe = pipeline.classes_[probs.argmax()]
    return classe, float(max_prob), []

testes = [
    'criar API REST com Ruby on Rails e PostgreSQL',
    'componente React com animação CSS e validação de formulário',
    'infraestrutura na AWS com Terraform e Docker',
    'algoritmo de machine learning com scikit-learn para classificação',
    'texto aleatório sobre culinária e receitas'
]

for t in testes:
    cat, prob, _ = predict_with_threshold(preprocess(t))
    print(f'{cat} ({prob:.4f}): {t}')

In [ ]:
joblib.dump(pipeline, MODEL_PATH)
print(f'Modelo salvo em: {MODEL_PATH}')
print(f'Categorias: {list(pipeline.classes_)}')